# Clean and balance the PhiUSIIL URL dataset
Run this notebook before `training.ipynb`. Source label `0` means phishing; saved `is_phishing = 1` means phishing. All non-filename source columns remain for auditing, though training uses only URL and label.

In [ ]:
"""Data preparation and URL-only three-layer phishing inference."""

from __future__ import annotations

import csv
import hashlib
import ipaddress
import math
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

import joblib
import numpy as np
import pandas as pd
import tldextract
from rapidfuzz import process
from rapidfuzz.distance import DamerauLevenshtein
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.frozen import FrozenEstimator
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline


EXTRACTOR = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=True)
ARTIFACT_VERSION = 1


def normalize_url(value: str) -> tuple[str, str, str]:
    """Return (normalized URL, hostname, registrable domain); never fetch the URL."""
    if not isinstance(value, str) or not value.strip():
        raise ValueError("URL must be a nonempty string")
    raw = value.strip()
    if any(ord(char) < 33 or ord(char) == 127 for char in raw):
        raise ValueError("URL contains whitespace or control characters")
    if "://" not in raw:
        raw = "https://" + raw
    try:
        parts = urlsplit(raw)
        if parts.scheme.lower() not in {"http", "https"} or not parts.hostname:
            raise ValueError("Only HTTP(S) URLs with a hostname are supported")
        hostname = parts.hostname.rstrip(".").encode("idna").decode("ascii").lower()
        if not hostname or any(not label or len(label) > 63 for label in hostname.split(".")):
            raise ValueError("Invalid hostname")
        ip_address = None
        try:
            ip_address = ipaddress.ip_address(hostname)
        except ValueError:
            if any(not all(c.isalnum() or c == "-" for c in label) or label[0] == "-" or label[-1] == "-" for label in hostname.split(".")):
                raise ValueError("Invalid hostname") from None
            if "." not in hostname:
                raise ValueError("Hostname must have a public suffix")
        port = parts.port  # Force validation of malformed ports.
        auth = f"{hostname}:{port}" if port is not None else hostname
        if ":" in hostname:  # IPv6 host in a URL requires brackets.
            auth = f"[{hostname}]" + (f":{port}" if port is not None else "")
        normalized = urlunsplit((parts.scheme.lower(), auth, parts.path or "/", parts.query, parts.fragment))
        if ip_address is not None:
            domain = hostname
        else:
            extracted = EXTRACTOR(hostname)
            if not extracted.suffix:
                # An unfamiliar but syntactically valid TLD still needs a stable group.
                domain = ".".join(hostname.split(".")[-2:])
            else:
                domain = f"{extracted.domain}.{extracted.suffix}" if extracted.domain else hostname
        return normalized, hostname, domain
    except (UnicodeError, ValueError) as exc:
        raise ValueError(f"Invalid URL: {value!r}: {exc}") from exc


def source_sha256(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def length_summary(frame: pd.DataFrame) -> pd.DataFrame:
    sample = frame.assign(url_length=frame["URL"].str.len())
    return sample.groupby("is_phishing")["url_length"].agg(
        ["count", "min", "median", "mean", "max"]
    ).round(2)


def clean_dataset(source_path: str | Path, output_path: str | Path, seed: int) -> tuple[pd.DataFrame, dict]:
    """Clean once, then balance classes without fabricating or dropping long phishing URLs."""
    frame = pd.read_csv(source_path, encoding="utf-8-sig", low_memory=False)
    frame.columns = frame.columns.str.lstrip("\ufeff").str.strip()
    if frame.columns.duplicated().any() or not {"FILENAME", "URL", "label"}.issubset(frame.columns):
        raise ValueError("Unexpected source CSV columns")
    original_rows = len(frame)
    frame = frame.drop(columns="FILENAME")
    frame["URL"] = frame["URL"].astype("string").str.strip()
    if frame["URL"].isna().any():
        raise ValueError("Missing URL")
    frame["is_phishing"] = frame.pop("label").map({0: 1, 1: 0})
    if frame["is_phishing"].isna().any():
        raise ValueError("Source label must be 0 or 1")
    frame["is_phishing"] = frame["is_phishing"].astype("int8")
    invalid = []
    for index, url in frame["URL"].items():
        try:
            normalize_url(url)
        except ValueError:
            invalid.append(index)
    if invalid:
        raise ValueError(f"{len(invalid)} invalid URLs; first row index: {invalid[0]}")
    conflicts = frame.groupby("URL")["is_phishing"].nunique()
    if (conflicts > 1).any():
        raise ValueError(f"{int((conflicts > 1).sum())} URLs have conflicting labels")
    deduplicated = frame.drop_duplicates(subset="URL", keep="first").copy()
    before = deduplicated["is_phishing"].value_counts().sort_index().to_dict()
    count = min(before.values())
    balanced = pd.concat(
        [part.sample(n=count, random_state=seed) for _, part in deduplicated.groupby("is_phishing")],
        ignore_index=True,
    ).sample(frac=1, random_state=seed).reset_index(drop=True)
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    balanced.to_csv(output, index=False, encoding="utf-8")
    report = {
        "source_rows": original_rows,
        "duplicate_urls_removed": original_rows - len(deduplicated),
        "unique_class_counts": before,
        "balanced_class_counts": balanced["is_phishing"].value_counts().sort_index().to_dict(),
        "before_length_summary": length_summary(deduplicated),
        "after_length_summary": length_summary(balanced),
    }
    return balanced, report


def load_ranked_domains(path: str | Path) -> list[str]:
    """Read rank,domain CSV without a header and retain its original order."""
    domains = []
    seen = set()
    with Path(path).open("r", encoding="utf-8", newline="") as handle:
        for row in csv.reader(handle):
            if len(row) < 2:
                continue
            domain = row[1].strip().rstrip(".").lower()
            if domain and domain not in seen:
                domains.append(domain)
                seen.add(domain)
    return domains


def split_by_domain(frame: pd.DataFrame, seed: int, test_size: float, validation_size: float):
    if not (0 < test_size < 1 and 0 < validation_size < 1 and test_size + validation_size < 1):
        raise ValueError("Invalid split sizes")
    groups = frame["URL"].map(lambda url: normalize_url(url)[2]).to_numpy()
    labels = frame["is_phishing"].to_numpy()
    outer_folds = round(1 / test_size)
    if outer_folds < 2 or not math.isclose(1 / outer_folds, test_size):
        raise ValueError("test_size must be a reciprocal integer, e.g. 0.20")
    outer = StratifiedGroupKFold(n_splits=outer_folds, shuffle=True, random_state=seed)
    remaining, test = next(outer.split(frame, labels, groups))
    relative_validation = validation_size / (1 - test_size)
    inner_folds = round(1 / relative_validation)
    if inner_folds < 2 or not math.isclose(1 / inner_folds, relative_validation):
        raise ValueError("validation_size must produce reciprocal folds, e.g. 0.10 with test_size 0.20")
    inner = StratifiedGroupKFold(n_splits=inner_folds, shuffle=True, random_state=seed + 1)
    train_local, val_local = next(inner.split(frame.iloc[remaining], labels[remaining], groups[remaining]))
    train, validation = remaining[train_local], remaining[val_local]
    group_sets = [set(groups[idx]) for idx in (train, validation, test)]
    assert not group_sets[0] & group_sets[1]
    assert not group_sets[0] & group_sets[2]
    assert not group_sets[1] & group_sets[2]
    assert not set(frame.iloc[train]["URL"]) & set(frame.iloc[test]["URL"])
    return train, validation, test, groups


def train_url_model(frame: pd.DataFrame, train: np.ndarray, validation: np.ndarray, config: dict):
    vectorizer = TfidfVectorizer(
            analyzer="char",
            ngram_range=tuple(config["ngram_range"]),
            sublinear_tf=True,
            norm="l2",
            min_df=config["min_df"],
            max_features=config["max_features"],
            dtype=np.float32,
    )
    urls = frame["URL"].map(lambda url: normalize_url(url)[0])
    labels = frame["is_phishing"].to_numpy()
    x_train = vectorizer.fit_transform(urls.iloc[train])
    x_validation = vectorizer.transform(urls.iloc[validation])
    y_train, y_validation = labels[train], labels[validation]
    alpha = float(config.get("alpha", 1.0 / (config["regularization_c"] * max(len(train), 1))))
    classifier = SGDClassifier(
        loss="log_loss", alpha=alpha, learning_rate=config.get("learning_rate", "optimal"),
        random_state=config["seed"], average=True,
    )
    batch_size = int(config.get("batch_size", 2048))
    epochs = int(config.get("epochs", 10))
    if epochs < 1 or batch_size < 1:
        raise ValueError("epochs and batch_size must be positive")
    rng = np.random.default_rng(config["seed"])
    for epoch in range(1, epochs + 1):
        order = rng.permutation(x_train.shape[0])
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            classifier.partial_fit(x_train[batch], y_train[batch], classes=np.array([0, 1]))
        validation_scores = classifier.predict_proba(x_validation)[:, 1]
        train_scores = classifier.predict_proba(x_train)[:, 1]
        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_loss={log_loss(y_train, train_scores):.4f} "
            f"val_loss={log_loss(y_validation, validation_scores):.4f} "
            f"val_accuracy={accuracy_score(y_validation, validation_scores >= 0.5):.4f}",
            flush=True,
        )
    pipeline = make_pipeline(vectorizer, classifier)
    calibrated = CalibratedClassifierCV(FrozenEstimator(pipeline), method="sigmoid")
    calibrated.fit(urls.iloc[validation], labels[validation])
    return calibrated


def score_metrics(y_true, scores, threshold: float) -> dict:
    predicted = np.asarray(scores) >= threshold
    return {
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "pr_auc": float(average_precision_score(y_true, scores)),
        "brier": float(brier_score_loss(y_true, scores)),
        "confusion_matrix": confusion_matrix(y_true, predicted, labels=[0, 1]).tolist(),
    }


class ThreeLayerDetector:
    def __init__(self, model, ranked_domains: list[str], config: dict):
        self.model = model
        self.ranked_domains = ranked_domains
        self.whitelist = set(ranked_domains)
        self.config = config.copy()
        self._typo_index_key = None
        self._typo_by_length = {}

    def _is_whitelisted(self, host: str) -> str | None:
        labels = host.split(".")
        for i in range(len(labels)):
            candidate = ".".join(labels[i:])
            if candidate in self.whitelist:
                return candidate
        return None

    def _ensure_typo_index(self):
        key = (int(self.config["typo_top_n"]), int(self.config["min_protected_name_length"]),
               float(self.config["typo_similarity_threshold"]))
        if key == self._typo_index_key:
            return
        n, minimum, _ = key
        if not 0 <= n <= len(self.ranked_domains):
            raise ValueError("typo_top_n must be between 0 and the whitelist size")
        by_length = defaultdict(list)
        for domain in self.ranked_domains[:n]:
            name = EXTRACTOR(domain).domain
            if len(name) >= minimum:
                by_length[len(domain)].append(domain)
        self._typo_by_length = dict(by_length)
        self._typo_index_key = key
        self._closest_typo.cache_clear()

    @lru_cache(maxsize=100_000)
    def _closest_typo(self, domain: str):
        self._ensure_typo_index()
        threshold = float(self.config["typo_similarity_threshold"])
        if not 0 < threshold <= 1:
            raise ValueError("typo_similarity_threshold must be in (0, 1]")
        length = len(domain)
        best = None
        for candidate_length in range(math.ceil(threshold * length), math.floor(length / threshold) + 1):
            choices = self._typo_by_length.get(candidate_length, ())
            if choices:
                found = process.extractOne(
                    domain, choices, scorer=DamerauLevenshtein.normalized_similarity,
                    score_cutoff=threshold,
                )
                if found and (best is None or found[1] > best[1]):
                    best = found
        return (best[0], float(best[1])) if best else None

    def explain(self, url: str) -> dict:
        normalized, host, domain = normalize_url(url)
        whitelisted = self._is_whitelisted(host)
        if whitelisted:
            return {
                "layer": "whitelist", "matched_domain": whitelisted, "similarity": None,
                "model_score": None, "final_score": 0.0,
                "reason": "Hostname is a listed domain or its subdomain; whitelist policy overrides the model.",
            }
        self._ensure_typo_index()
        match = self._closest_typo(domain)
        if match:
            return {
                "layer": "typosquat", "matched_domain": match[0], "similarity": match[1],
                "model_score": None, "final_score": 1.0,
                "reason": "Registrable domain resembles a protected top-N domain; 1.0 is a policy override, not a calibrated probability.",
            }
        score = float(self.model.predict_proba([normalized])[0, 1])
        return {
            "layer": "model", "matched_domain": None, "similarity": None,
            "model_score": score, "final_score": score,
            "reason": "Neither domain rule matched; score is the calibrated URL model probability.",
        }

    def predict_proba(self, url: str) -> float:
        return float(self.explain(url)["final_score"])


def save_bundle(path: str | Path, detector: ThreeLayerDetector, metadata: dict):
    output = Path(path)
    output.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {"artifact_version": ARTIFACT_VERSION, "model": detector.model,
         "ranked_domains": detector.ranked_domains, "config": detector.config,
         "label_mapping": {"source_0": "phishing", "source_1": "legitimate", "is_phishing_1": "phishing"},
         "metadata": metadata},
        output,
        compress=3,
    )


def load_bundle(path: str | Path) -> tuple[ThreeLayerDetector, dict]:
    bundle = joblib.load(path)
    if bundle.get("artifact_version") != ARTIFACT_VERSION:
        raise ValueError("Unsupported artifact version")
    return ThreeLayerDetector(bundle["model"], bundle["ranked_domains"], bundle["config"]), bundle["metadata"]


In [ ]:
from pathlib import Path
from IPython.display import display

SOURCE_PATH = Path('dataset/PhiUSIIL_Phishing_URL_Dataset.csv')
CLEANED_PATH = Path('dataset/cleaned_balanced.csv')
RANDOM_SEED = 42


In [ ]:
cleaned, report = clean_dataset(SOURCE_PATH, CLEANED_PATH, RANDOM_SEED)
print(f"Source rows: {report['source_rows']:,}")
print(f"Duplicate URLs removed: {report['duplicate_urls_removed']:,}")
print('Unique class counts before balancing:', report['unique_class_counts'])
print('Balanced class counts:', report['balanced_class_counts'])
print('URL length by class before balancing:')
display(report['before_length_summary'])
print('URL length by class after balancing:')
display(report['after_length_summary'])
assert cleaned['is_phishing'].value_counts().nunique() == 1
assert 'FILENAME' not in cleaned.columns and 'label' not in cleaned.columns
assert not cleaned['URL'].duplicated().any()
print(f'Saved {len(cleaned):,} rows to {CLEANED_PATH}')
